# v28 — fully verified T4 candidate

80.75 → 44.43 seconds; 7153 MiB; all ten checks passed.
Exact measured source `4aaaade07f255269fb8ef8cf7a6b6007d95120b7` and raw reports were hash-verified.
Select GPU T4 x2, Internet On, then Run All. The source stays pinned; rerun labels are unique.


In [ ]:
import uuid
REPO_URL = "https://github.com/promaladoi/Decentrathon-6.0.git"
COMMIT = "4aaaade07f255269fb8ef8cf7a6b6007d95120b7"
VERSION = "v28_repeat_" + uuid.uuid4().hex[:8]


In [ ]:
import subprocess, os, json, zipfile, uuid
from pathlib import Path
from IPython.display import display, FileLink
print(subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],text=True))
subprocess.run(['python','-m','pip','install','-q','soup-cli[train]==0.77.0','transformers==5.16.1','peft==0.20.0','trl==0.29.1','bitsandbytes==0.50.2','accelerate==1.14.0','datasets==4.8.5','pytest'],check=True)

from importlib.metadata import version as package_version
assert package_version('torch').split('+')[0] == '2.11.0', 'Use the original Kaggle Torch 2.11 environment for this pinned T4 candidate'
assert package_version('triton') == '3.6.0', 'This measured candidate uses Triton 3.6.0'


In [ ]:
assert not subprocess.check_output(['nvidia-smi','--query-compute-apps=pid','--format=csv,noheader'],text=True).strip(), 'GPU is busy: stop the other GPU work before running this notebook.'
p=Path('/kaggle/working')/('nurzhigit_'+VERSION+'_'+uuid.uuid4().hex[:8])
subprocess.run(['git','clone','-q',REPO_URL,str(p)],check=True,timeout=120)
subprocess.run(['git','-C',str(p),'checkout','-q',COMMIT],check=True)
print('MEASURED CODE:',subprocess.check_output(['git','-C',str(p),'rev-parse','HEAD'],text=True).strip())
subprocess.run(['python','task/prepare_data.py'],cwd=p,check=True)


In [ ]:
completed = subprocess.run(['python','04_submission/run_candidate.py','--version',VERSION],cwd=p)
display(FileLink('codex_'+VERSION+'.zip'))
assert completed.returncode == 0, 'Experiment stopped or rejected; inspect its output/archive.'
